# Cognopolis · M6.5 — Воркспейс: день шахтёра

Наверху добычу гейтит **время** (кулдаун), в Подземье — **код**: медь достаётся за решённое
Испытание — маленькую задачу класса GAIA над настоящими файлами (записка + CSV-гроссбух).
Ошибся — минус hp; пять промахов на серию; смерть жжёт весь вьюк. «Шанс смерти» здесь —
частота ошибок вашего кода, а не кубик.

**Что в этом воркспейсе.** Весь забег end-to-end: дорога до входа → спуск → навигация по
горизонту (как в M1) → испытание жилы → решатель из трёх строк → подъём с фиксацией →
зачисление руды → авто-банк дома. В конце на складе лежит первая медь.

> **Working-first.** `Run all` проходит целиком без правок. Нужен только токен своего
> аккаунта (секрет `COGNOPOLIS_TOKEN`); LLM-ключ и золото не нужны — решатель
> детерминирован, аренда входа в срезе 1 равна нулю. Демо-аккаунт `testuser` в шахту не
> пускает демо-гард — зарегистрируйте свой.

**Ссылки:** лекция урока — сайт курса, «Подземье: агент-рудокоп (M6.5)» ·
API-доки: `https://kindomklaster.com/docs` (кнопка Authorize) ·
кредо Подземья — сначала пройдите одну печать РУКАМИ в игре: тайл входа →
«Спуститься в шахту» → на жиле панель «Копать» (в ней же блок «что понадобится агенту»).

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis-client.git"

In [ ]:
import os, time
from cognopolis_client import Client, GameError

# ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Свой сервер — через переменную COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

def get_secret(name: str) -> str:
    """Секрет из env / Colab / Kaggle — одним хелпером (канон курса)."""
    val = os.environ.get(name, "")
    if val:
        return val
    try:
        from google.colab import userdata            # Colab: значок ключа слева -> Secrets
        return userdata.get(name) or ""
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle: Add-ons -> Secrets
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        pass
    return ""

# Токен жителя: Ратуша -> вкладка «аккаунт» -> «копировать». Демо-аккаунт testuser в шахту
# не пускает демо-гард (demo_forbidden) — нужен свой аккаунт.
TOKEN = get_secret("COGNOPOLIS_TOKEN")
assert TOKEN, (f"Нужен токен жителя мира {BASE_URL} — секрет/переменная COGNOPOLIS_TOKEN "
               "(Ратуша -> вкладка «аккаунт»).")

c = Client(BASE_URL, token=TOKEN)

# Мягкая проверка связи: если мир недоступен — живые ячейки ниже честно пропустятся.
WORLD_UP = True
try:
    c.get_map()                       # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — "
          "проверь COGNOPOLIS_URL или попробуй позже.")

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)
print("Смотреть за агентом в браузере:", f"{BASE_URL}/?token={TOKEN}")

## Разведка — вход на карте мира и карта горизонта

Вход в Подземье — обычный тайл `mine_entrance` на общей карте. Карта горизонта — публичная
(`GET /mine/map`) и статичная: entry (вход = выход), жилы и ствол глубже. Заряды жил — общие
на всех шахтёров мира.

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    print(f"Житель: ({ch['x']},{ch['y']}), hp {ch['hp']}, под землёй: {ch.get('in_mine', False)}")

    # Вход в Подземье — обычный тайл на карте мира; ищем его, а не хардкодим.
    world = c.get_map()
    ENTRANCE = next((t["x"], t["y"]) for t in world["tiles"]
                    if t.get("content") == "mine_entrance")
    print("Вход в Подземье на карте мира:", ENTRANCE)

    # Карта горизонта 1 — публичная и статичная (меняется только состояние жил): кэшируй смело.
    mm = c.mine_map(horizon=1)
    GLYPH = {"entry": "E", "descent": "D", "vein": "\u2116", "empty": "\u00b7"}
    grid = [["\u00b7"] * mm["size"] for _ in range(mm["size"])]
    for t in mm["tiles"]:
        grid[t["y"]][t["x"]] = GLYPH.get(t["content"], "?")
    print("\nГоризонт 1 (E — вход/выход, \u2116 — жила, D — ствол глубже):")
    for row in grid:
        print("  " + " ".join(row))

    VEINS = [(t["x"], t["y"]) for t in mm["tiles"] if t["content"] == "vein"]
    MINE_ENTRY = next((t["x"], t["y"]) for t in mm["tiles"] if t["content"] == "entry")
    print(f"\nЖилы горизонта 1: {VEINS}; entry (вход = выход): {MINE_ENTRY}")
else:
    print("мир недоступен — пропуск")

## Навигация — те же 8 направлений, что в M1

Перемещение и карта под землёй — контракт-зеркала поверхности; новых «кнопок движения» нет.
Единственная разница — конверт: сервисные действия `/mine/*` возвращают
`{result, cooldown, miner}`, а не `{result, cooldown, character}`. Кулдаун в корне обоих,
поэтому `wait_cooldown()` работает одинаково.

In [ ]:
# 8 направлений — как в M1 (D-069); диагонали экономят ходы. Так ходят И наверху, И в шахте:
# навигация — общая механика, «вообще по-другому» под землёй работает только добыча.
STEP_DIR = {(1, 0): "east", (-1, 0): "west", (0, 1): "south", (0, -1): "north",
            (1, 1): "southeast", (-1, 1): "southwest",
            (1, -1): "northeast", (-1, -1): "northwest"}

def sign(v: int) -> int:
    return (v > 0) - (v < 0)

def walk_to(x: int, y: int, why: str = "") -> dict:
    """Шагать по ПОВЕРХНОСТИ до (x, y). Конверт игры: {result, cooldown, character}."""
    ch = c.get_character()
    while (ch["x"], ch["y"]) != (x, y):
        step = (sign(x - ch["x"]), sign(y - ch["y"]))
        r = c.move_dir(STEP_DIR[step], reason=why)
        ch = r["character"]
        if r["result"].get("banked"):
            print("  дом: авто-банк", r["result"]["banked"])
        c.wait_cooldown()
    return ch

def mine_walk_to(x: int, y: int, why: str = "") -> dict:
    """Шагать по ГОРИЗОНТУ ШАХТЫ до (x, y). Конверт сервиса: {result, cooldown, miner}."""
    m = c.mine_observe()
    while (m["x"], m["y"]) != (x, y):
        step = (sign(x - m["x"]), sign(y - m["y"]))
        r = c.mine_move(STEP_DIR[step], reason=why)
        m = r["miner"]
        c.wait_cooldown()
    return m

print("Навигация готова: walk_to (поверхность) и mine_walk_to (шахта).")

## Решатель горизонта 1 — «счётные печати»

Испытание отдаёт **настоящие файлы**: `note.txt` (записка: какой товар и какой месяц) и
`ledger.csv` (гроссбух). Ответ — сумма `количество × цена` по строкам, где совпали И товар,
И месяц. Слать наугад — самоубийство (бюджет 5 промахов, промах = минус hp); пересылать
файлы LLM — из пушки по воробьям. Три строки детерминированного кода — это и есть **Приём**:
единственное, что рогалик не отбирает даже смертью.

In [ ]:
import csv, io, re

# Правила датасета горизонта 1: записка называет товар в «ёлочках» (единственная такая пара
# в тексте) и месяц — одним словом с заглавной после слова «месяц».
GOOD_RE = re.compile(r"«([^»]+)»")
MONTH_RE = re.compile(r"месяц\s+([А-ЯЁ][а-яё]+)")

def solve(files: dict[str, bytes]) -> str:
    """Приём горизонта 1: «счётные печати». Читает записку, фильтрует гроссбух по товару
    И месяцу, возвращает сумму количество*цена строкой — ровно то, что спрашивает печать."""
    note = files["note.txt"].decode("utf-8")
    good = GOOD_RE.search(note).group(1)
    month = MONTH_RE.search(note).group(1)
    total = sum(
        int(row["количество"]) * int(row["цена"])
        for row in csv.DictReader(io.StringIO(files["ledger.csv"].decode("utf-8")))
        if row["товар"] == good and row["месяц"] == month)
    return str(total)

# Приём — это код: его можно проверить БЕЗ шахты, на своих файлах.
demo_files = {
    "note.txt": "Смотритель велит счесть выручку за «гвозди» в месяц Ковеня.".encode(),
    "ledger.csv": ("товар,месяц,количество,цена\n"
                   "гвозди,Ковеня,3,5\n"
                   "гвозди,Ливеня,10,2\n"
                   "верёвка,Ковеня,7,4\n").encode(),
}
assert solve(demo_files) == "15", "3 гвоздя по 5 в Ковене = 15"
print("Решатель жив: demo-печать даёт", solve(demo_files))

## Дорога вниз

Спуск (`mine_descend`) работает только с тайла входа. Забег переживает обрыв ноутбука —
если вы уже под землёй, просто продолжаем (`mine_observe` скажет, где вы).

In [ ]:
if WORLD_UP:
    T_DAY = time.time()
    ch = c.get_character()
    if ch.get("in_mine"):
        # Забег переживает обрыв ноутбука: он висит на сервере, просто продолжаем.
        print("Уже под землёй — продолжаем начатый забег.")
    else:
        walk_to(*ENTRANCE, why="к входу в шахту")
        try:
            r = c.mine_descend(reason="первый забег за медью")   # аренда в срезе 1 = 0
            print("спуск:", r["result"])
            c.wait_cooldown()
        except GameError as e:
            if e.code != "in_mine":
                raise
            print("Сервер говорит: уже в шахте — продолжаем.")
    m = c.mine_observe()
    print(f"Горизонт {m['horizon']}, позиция ({m['x']},{m['y']}), hp {m['hp']}, "
          f"вьюк {m['satchel']} (cap {m['satchel_cap']})")
else:
    print("мир недоступен — пропуск")

## Петля добычи

К жиле → `mine_trial` → скачать файлы → `solve` → `mine_attempt(answer, series=...)`.
`series` передаём всегда: если жилу успели выбить и реставрировать, сервис ответит честным
`series_changed` ДО урона — это отказ-объяснение, не промах. Берём **два заряда** и уходим:
жилы общие, оставьте меди другим.

In [ ]:
if WORLD_UP:
    ORE_TARGET = 2      # вежливость к общему миру: пара зарядов за забег — и наверх
    m = c.mine_observe()
    mined = sum(m["satchel"].values())
    # Жилы обходим от ближней к дальней (метрика Чебышёва — диагонали бесплатные).
    for vein in sorted(VEINS, key=lambda v: max(abs(v[0] - m["x"]), abs(v[1] - m["y"]))):
        if mined >= ORE_TARGET:
            break
        m = mine_walk_to(*vein, why="к жиле")
        while mined < ORE_TARGET:
            try:
                trial = c.mine_trial()                    # вопрос + артефакты (бесплатно)
                answer = solve(c.mine_trial_files())      # Приём горизонта 1
                r = c.mine_attempt(answer, series=trial["progress"]["series"])
                c.wait_cooldown()
            except GameError as e:
                # vein_depleted / series_changed / satchel_full... — честные отказы общего
                # мира, НЕ ошибки кода: пауза и дальше (отказ ничего не жжёт).
                print(f"  жила {vein}: отказ {e.code} — иду дальше")
                time.sleep(1)
                break
            out, m = r["result"], r["miner"]
            if out["outcome"] == "hit":
                mined = sum(m["satchel"].values())
                print(f"  hit: +{out['ore']} — вьюк {m['satchel']}, hp {m['hp']}")
                if out.get("vein") == "depleted":
                    print(f"  жила {vein} погасла — реставрация придёт по таймеру")
                    break
            else:
                print(f"  miss: hp={out['hp']}, бюджет={out['budget_left']} — проверь решатель!")
                break
    print(f"Во вьюке после петли: {mined} руды")
else:
    print("мир недоступен — пропуск")

## Дорога наверх — точка невозврата и две волны фиксации

Подъём на entry (`mine_up`) — **финализация**: действия в шахте больше не работают, вьюк
стал манифестом. `mine_return` пересыпает руду в рюкзак (даже сверх вместимости — без
потерь), а шаг на дом (0,0) — знакомый авто-банк. Руда фиксируется дважды: вьюк → рюкзак →
склад; по дороге домой действует обычная жизнь поверхности.

In [ ]:
if WORLD_UP:
    try:
        m = mine_walk_to(*MINE_ENTRY, why="к стволу — фиксировать вьюк")
        r = c.mine_up(reason="подъём: финализация забега")
        print("финализация:", r["result"])       # {'outcome': 'exited', 'ore': {...}}
        c.wait_cooldown()
    except GameError as e:
        if e.code not in {"invalid_token", "run_not_active"}:
            raise
        print("Забег уже финализирован — сразу к зачислению.")
    try:
        r = c.mine_return(reason="зачисление руды")
        print("зачисление:", r["result"])        # руда манифеста -> рюкзак, без потерь
        c.wait_cooldown()
    except GameError as e:
        if e.code not in {"not_in_mine", "nothing_to_settle"}:
            raise
        print(f"Зачислять нечего ({e.code}) — видимо, уже рассчитаны.")
    ch = walk_to(0, 0, why="домой — сдать медь на склад")
    print("рюкзак:", ch["inventory"], "| склад copper:", ch["stored"].get("copper", 0))
else:
    print("мир недоступен — пропуск")

## Проверка

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    copper = ch["stored"].get("copper", 0)
    assert copper >= 1, (
        "На складе нет меди. Если в логе выше сплошные vein_depleted — жилы выбиты другими "
        "шахтёрами: подожди реставрации (минуты) и перезапусти петлю добычи.")
    print(f"Медь на складе: {copper}. Первая медь Подземья добыта и вынесена — критерий урока выполнен.")
    print(f"День шахтёра занял {time.time() - T_DAY:.0f}с чистого времени.")
else:
    print("мир недоступен — пропуск проверки")

## Наблюдаемость и что дальше

Откройте `BASE_URL/?token=<ваш токен>` в браузере: тумблер «карта/шахта» покажет жителя под
землёй, журнал забега — каждый hit/miss, а Хроника — вехи «спустился/вышел с грузом». Пока
агент решает печать, детектор застоя честно пишет «Решает печать…» — думать легально.

Дальше по курсу — домашка `notebook.ipynb` (соберите день шахтёра сами: маршрут, политика
push-your-luck, робастность к отказам). Глубже в игре — гроссбухи, руны и Библиотека Приёмов
(следующие срезы Подземья), а сиды испытаний делают шахту полигоном для evals (M7).